# CS116 S6E10 — Power v4: Safe Dual Teachers + Diverse Learner + Original Augmentation

**Mốc cần so sánh:** `quan_xgb_route_teacher_v3`: **0.960940987539** Overall OOF ROC-AUC (5 folds).

4 **member độc lập**, không blend, không tự submit:
1. `quan_xgb_safe_teacher_v4`: tái lập v3 sau khi **loại original rows trùng features** với train/test cuộc thi — control/audit.
2. `quan_xgb_dual_teacher_v4`: XGB student + hai original-only teachers (XGB & CatBoost) — thử bổ sung tín hiệu.
3. `quan_cat_dual_teacher_v4`: CatBoost GPU student + hai teachers — diversity khác thuật toán.
4. `quan_xgb_original_aug_v4`: thêm original labeled rows vào **chỉ training side** của outer fold, giảm trọng số original còn 0.35, **không dùng teacher in-sample**.

Cả bốn đều dùng shared `folds.csv` qua `data.py`; 5-fold cross-fitted target encoding; early stop bằng 10% của training side, không nhìn nhãn outer fold để dừng cây. Mỗi fold tự lưu checkpoint, cuối cùng tạo 8 `.npy` và ZIP.

**Nguồn Add Input:** Competition S6E10, `quannh206/fold-data`, notebook output `quannh206/cs116`, `arseniyshutko/binary-aviation-satisfaction-129k`. Bật Kaggle GPU T4.

**Lưu ý:** Đây là thí nghiệm có giả thuyết, **không bảo đảm tăng điểm**. Nếu audit phát hiện nhiều trùng lặp, kiểm tra trước khi dùng kết quả v3. `Save Version → Save & Run All` chạy lại toàn bộ notebook. Tham chiếu `STACKING.md` của nhóm.

In [ ]:
from pathlib import Path
import sys, os, gc, csv, time, json, shutil, zipfile, subprocess, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# ===== V4: SAFE ORIGINAL + DIVERSITY + AUGMENTATION =====
RUN_TEACHER_MEMBER = False
RUN_XGB_TEACHER = False
RUN_XGB_ROUTE_TEACHER = False
RUN_LGB_ROUTE_TEACHER = False  # LightGBM dùng CPU (tạo diversity), ước lượng chạy lâu hơn XGB
TEACHER_SEEDS = [13, 2026]   # model chỉ học original dataset
ORIGINAL_DATASET_DIR = ''   # tùy chọn: đường dẫn chính xác nếu auto-detect không ra

MAX_TREES = 2600
EARLY_STOP_ROUNDS = 120
WORK_PROJECT = Path('/kaggle/working/cs116-kaggle-s6e10-power-v4')
WORK_SRC = WORK_PROJECT/'src'
DATA_DIR = WORK_PROJECT/'data'
OUT_DIR = WORK_SRC/'oof'
PROGRESS_DIR = WORK_PROJECT/'progress_power_v4'
OUT_ZIP = Path('/kaggle/working/S6E10_Power_v4_members.zip')
MANIFEST = Path('/kaggle/working/S6E10_Power_v4_manifest.json')
for p in (WORK_PROJECT, WORK_SRC, DATA_DIR, OUT_DIR, PROGRESS_DIR):
    p.mkdir(parents=True, exist_ok=True)

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    import lightgbm as lgb
    from sklearn.preprocessing import TargetEncoder
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
except ImportError as e:
    raise RuntimeError('Required packages missing. Use Kaggle default image with recent xgboost/sklearn.') from e

print('XGBoost', xgb.__version__)
import sklearn
print('scikit-learn', sklearn.__version__)
assert hasattr(sklearn.preprocessing, 'TargetEncoder'), 'Need sklearn >= 1.3'
print('GPU check:')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],check=True)
# Fail fast: GPU is a requirement, not an invisible CPU fallback.
import torch
assert torch.cuda.is_available(), 'Bật GPU T4 trong Kaggle Settings trước khi Save & Run All'
print('GPU available:',torch.cuda.get_device_name(0))


# Chạy cả 4 member. Bạn có thể tắt member cuối nếu muốn giảm thời gian.
RUN_SAFE_BASE = True
RUN_XGB_DUAL = True
RUN_CAT_DUAL = True
RUN_XGB_ORIGAUG = True
ORIG_WEIGHT = 0.35  # Mức đóng góp ban đầu của original rows, cố định trước CV
REFERENCE_V3 = 0.9609409875392086

In [ ]:
# ===== DÙNG ĐÚNG TEAM load_data() / folds.csv =====
PROJECT_INPUT = Path('/kaggle/input/notebooks/quannh206/cs116/cs116-kaggle-s6e10')
ORIGINALS = {
    'train.csv': Path('/kaggle/input/competitions/playground-series-s6e10/train.csv'),
    'test.csv': Path('/kaggle/input/competitions/playground-series-s6e10/test.csv'),
    'folds.csv': Path('/kaggle/input/datasets/quannh206/fold-data/folds.csv'),
}
for name, path in ORIGINALS.items():
    assert path.is_file(), f'Add Input: {path}'
    target = DATA_DIR/name
    if target.is_symlink(): target.unlink()
    elif target.exists():
        # Keep only a correct previous link; never silently read stale actual data.
        raise FileExistsError(f'Unexpected real file at {target}; use a clean Kaggle session')
    target.symlink_to(path)
assert (PROJECT_INPUT/'src'/'data.py').is_file(), f'Add Input: old CS116 notebook output ({PROJECT_INPUT})'
shutil.copytree(PROJECT_INPUT/'src', WORK_SRC, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__','*.pyc','oof'))
for mod in ('data','utils','config'): sys.modules.pop(mod,None)
sys.path.insert(0,str(WORK_SRC))
import data as team_data
team_data.DATA_DIR = DATA_DIR
train,test = team_data.load_data()

fold_df = pd.read_csv(ORIGINALS['folds.csv'],usecols=['id','fold'])
assert train.shape==(699635,24),train.shape
assert test.shape==(299844,22),test.shape
assert np.array_equal(train.id.to_numpy(),fold_df.id.to_numpy())
assert np.array_equal(train.fold.to_numpy(),fold_df.fold.to_numpy())
assert set(train.fold.unique())==set(range(5))
assert train.id.is_unique and test.id.is_unique
Y=train['satisfaction'].astype(np.int32).to_numpy()
assert set(np.unique(Y))=={0,1}
FOLDS=train.fold.to_numpy(dtype=np.int8)
FEATURES=[c for c in test.columns if c!='id']
CATS=['Gender','Customer Type','Type of Travel','Class']
assert len(FEATURES)==21 and all(c in FEATURES for c in CATS)
NUMS=[c for c in FEATURES if c not in CATS]
print('Loaded via team data.py:',train.shape,test.shape)
print('Fold counts:',pd.Series(FOLDS).value_counts().sort_index().to_dict())

In [ ]:
# ===== BASE FEATURES + CONDITIONAL TE + ROUTE (NO FREQUENCY) =====
start_features = time.perf_counter()
raw = pd.concat([train[FEATURES], test[FEATURES]], ignore_index=True)
n_train = len(train)
base = pd.get_dummies(raw, columns=CATS, dummy_na=True, dtype=np.float32)
BASE_ARRAY = base.to_numpy(dtype=np.float32)
BASE_DIM = BASE_ARRAY.shape[1]
del base

# Original 21 target-encoded columns as exact categorical tokens.
TE_BASE = pd.DataFrame(index=raw.index)
for col in FEATURES:
    TE_BASE[col] = raw[col].astype('string').fillna('__MISSING__').astype(str)

# Exactly the same 16 conditional TE columns used in Context v1 / nofreq ablation.
TE_CONTEXT = TE_BASE.copy()
seg = TE_BASE['Type of Travel'] + '|' + TE_BASE['Customer Type'] + '|' + TE_BASE['Class']
TE_CONTEXT['segment_TCC'] = seg
SCORES = [
    'Inflight wifi service', 'Online boarding', 'Inflight entertainment',
    'Seat comfort', 'Ease of Online booking', 'Cleanliness',
    'On-board service', 'Leg room service', 'Checkin service', 'Baggage handling'
]
for col in SCORES:
    TE_CONTEXT['ctx_' + col] = seg + '|' + TE_BASE[col]
for col, group in [
    ('Flight Distance', 'Class'), ('Flight Distance', 'Type of Travel'),
    ('Flight Distance', 'Customer Type'), ('Age', 'Class'), ('Age', 'Type of Travel')
]:
    TE_CONTEXT['ctx_' + col + '|' + group] = TE_BASE[col] + '|' + TE_BASE[group]

# Exactly 13 unlabeled route statistics (same as Context v1 / nofreq ablation).
r = raw['Flight Distance']
route_cnt = r.groupby(r, dropna=False).transform('size').to_numpy(dtype=np.float32)
route_features = {'route_log_count': np.log1p(route_cnt).astype(np.float32)}
for col in [
    'Online boarding', 'Inflight wifi service', 'Seat comfort',
    'Inflight entertainment', 'Cleanliness', 'Ease of Online booking'
]:
    rating = raw[col].astype(np.float32)
    grp_sum = rating.groupby(r, dropna=False).transform('sum').to_numpy(dtype=np.float32)
    vals = rating.to_numpy(dtype=np.float32)
    glob = float(rating.mean())
    route_mean = np.where(route_cnt > 1,
                          (grp_sum - vals) / np.maximum(route_cnt - 1, 1),
                          glob).astype(np.float32)
    route_features['route_mean_' + col] = route_mean
    route_features['route_resid_' + col] = (vals - route_mean).astype(np.float32)
ROUTE_ARRAY = pd.DataFrame(route_features).to_numpy(dtype=np.float32)
assert ROUTE_ARRAY.shape[1] == 13


# ===== ROUTE PROFILE EXTENSION: label-free + leave-one-out =====
# Use all feature rows (train + test), never target label.
rich = {}
route_count = raw['Flight Distance'].groupby(raw['Flight Distance'], dropna=False).transform('size').to_numpy(dtype=np.float32)
route_key = raw['Flight Distance']
for col in ['Online boarding','Inflight wifi service','Seat comfort',
            'Inflight entertainment','Cleanliness','Ease of Online booking']:
    x = pd.to_numeric(raw[col],errors='coerce').astype(np.float32)
    xvals = x.to_numpy(dtype=np.float32)
    sums = x.groupby(route_key,dropna=False).transform('sum').to_numpy(dtype=np.float32)
    sq = x.pow(2).groupby(route_key,dropna=False).transform('sum').to_numpy(dtype=np.float32)
    den = np.maximum(route_count-1,1)
    mu = np.where(route_count>1,(sums-xvals)/den,float(x.mean())).astype(np.float32)
    var = np.where(route_count>2,(sq-xvals**2)/den-mu**2,0).astype(np.float32)
    rich['route_std_loo_'+col]=np.sqrt(np.maximum(var,0)).astype(np.float32)
    hi = (x>=4).astype(np.float32)
    hi_vals=hi.to_numpy(dtype=np.float32)
    hi_sum=hi.groupby(route_key,dropna=False).transform('sum').to_numpy(dtype=np.float32)
    rich['route_high_share_loo_'+col]=np.where(route_count>1,
                      (hi_sum-hi_vals)/den,float(hi.mean())).astype(np.float32)
for col in CATS:
    # other passengers at exactly same distance and category
    group_n = raw.groupby(['Flight Distance',col],dropna=False)[col].transform('size').to_numpy(dtype=np.float32)
    rich['route_samecat_share_loo_'+col]=np.where(route_count>1,
             (group_n-1)/np.maximum(route_count-1,1),0).astype(np.float32)
RICH_EXTRA = pd.DataFrame(rich).to_numpy(dtype=np.float32)
assert RICH_EXTRA.shape==(len(raw),16)
assert np.isfinite(RICH_EXTRA).all()
assert np.isfinite(ROUTE_ARRAY).all()
print('Base dim',BASE_DIM,'Context TE',TE_CONTEXT.shape[1],
      'Base route',ROUTE_ARRAY.shape[1],'Rich route',RICH_EXTRA.shape[1])
del rich, route_count, route_key, x, xvals, sums, sq, den, mu, var, hi, hi_vals, hi_sum

In [ ]:
# ===== TRAINING + CHECKPOINT + OUTPUT =====

def encoder(seed):
    return TargetEncoder(target_type='binary',smooth='auto',
                         cv=5,shuffle=True,random_state=seed)

def learner(n_estimators,seed,early_stop):
    return XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=0.035,
        max_depth=6,
        min_child_weight=10,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5.0,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        device='cuda:0',
        random_state=seed,
        n_jobs=4,
        early_stopping_rounds=EARLY_STOP_ROUNDS if early_stop else None,
    )

RUN_META={
 'reference_best_oof_auc':REFERENCE_V3,
 'purpose':'Leak-audited original-only dual teachers; XGB and CatBoost students; weighted original-row augmentation; independent OOF members',
 'project':'CS116 Kaggle S6E10',
 'shared_folds':'quannh206/fold-data/folds.csv',
 'started_utc':datetime.now(timezone.utc).isoformat(),
 'models':{}
}

# Tách ZIP khỏi các ZIP cũ, tránh ghi đè artifact của member trước.
KNOWN_OLD_ZIPS=[]

def package_results():
    # Important: incomplete progress is NOT published under member OOF names.
    MANIFEST.write_text(json.dumps(RUN_META,indent=2,ensure_ascii=False),encoding='utf-8')
    with zipfile.ZipFile(OUT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as dst:
        names=set()
        for old in KNOWN_OLD_ZIPS:
            if not old.is_file(): continue
            with zipfile.ZipFile(old) as src:
                for filename in src.namelist():
                    if filename.endswith('.npy') and ('oof_' in filename or 'test_' in filename):
                        bn=Path(filename).name
                        if bn not in names:
                            dst.writestr(bn,src.read(filename))
                            names.add(bn)
            print('Included previous predictions from:',old)
            break
        for artifact in sorted(OUT_DIR.glob('*.npy')):
            if artifact.name not in names:
                dst.write(artifact,arcname=artifact.name)
                names.add(artifact.name)
        dst.write(MANIFEST,arcname=MANIFEST.name)
        exp=WORK_PROJECT/'experiments_power_v4.csv'
        if exp.exists():dst.write(exp,arcname=exp.name)
    print('Updated final ZIP:',OUT_ZIP,'files:',len(names),'size(MB)',round(OUT_ZIP.stat().st_size/1e6,2))

def progress_paths(name):
    return dict(oof=PROGRESS_DIR/(name+'_partial_oof.npy'),
                tsum=PROGRESS_DIR/(name+'_partial_testsum.npy'),
                meta=PROGRESS_DIR/(name+'.json'))

def open_progress(name,config):
    paths=progress_paths(name)
    if paths['meta'].exists():
        old=json.loads(paths['meta'].read_text())
        assert old['config']==config, 'Config changed but member name reused!'
        arr=np.load(paths['oof']);tsum=np.load(paths['tsum'])
        done=list(map(int,old['done']))
        assert arr.shape==(len(train),) and tsum.shape==(len(test),)
        return arr,tsum,done,old.get('fold_details',[])
    return np.full(len(train),np.nan,dtype=np.float64),np.zeros(len(test),dtype=np.float64),[],[]

def save_progress(name,config,oof,tsum,done,details):
    paths=progress_paths(name)
    np.save(paths['oof'],oof)
    np.save(paths['tsum'],tsum)
    paths['meta'].write_text(json.dumps({'config':config,'done':sorted(done),'fold_details':details},indent=2))
    RUN_META['models'][name]={'complete':False,'folds_done':sorted(done),'fold_details':details}
    package_results() # keep a ZIP and manifest after each fold (partial OOF NOT published).

def official_save(name,oof,testp):
    assert oof.shape==(699635,) and testp.shape==(299844,)
    assert np.isfinite(oof).all() and np.isfinite(testp).all()
    assert np.all((oof>=0)&(oof<=1)) and np.all((testp>=0)&(testp<=1))
    np.save(OUT_DIR/f'oof_{name}.npy',oof.astype(np.float64))
    np.save(OUT_DIR/f'test_{name}.npy',testp.astype(np.float64))
    if (WORK_SRC/'utils.py').is_file():
        import utils as u
        # Team function checks shapes/AUC and stores to team output folder.
        if callable(getattr(u,'save_preds',None)):
            cwd=os.getcwd()
            try:
                os.chdir(WORK_SRC)
                u.save_preds(name,oof,testp,Y)
            finally:
                os.chdir(cwd)

def experiment_row(name,auc,notes):
    file=WORK_PROJECT/'experiments_power_v4.csv'
    header=['date','person','name','cv_auc','lb_auc','notes']
    rows=[]
    if file.exists():
        with file.open(newline='',encoding='utf-8') as f:
            rows=list(csv.reader(f))[1:]
    rows=[r for r in rows if len(r)<3 or r[2]!=name]
    rows.append([datetime.now(timezone.utc).strftime('%Y-%m-%d'),'quan',name,f'{auc:.9f}','',notes])
    with file.open('w',newline='',encoding='utf-8') as f:
        w=csv.writer(f);w.writerow(header);w.writerows(rows)

def fold_summary(oof):
    return [float(roc_auc_score(Y[FOLDS==i],oof[FOLDS==i])) for i in range(5)]

# Predict with GPU array where CuPy is installed; fallback to numpy (warning is benign).
try:
    import cupy as cp
    HAS_CUPY=True
except ImportError:
    HAS_CUPY=False

def predict_proba(model,X):
    if HAS_CUPY:
        try:
            gpu=cp.asarray(X)
            prob=model.predict_proba(gpu)[:,1]
            ret=cp.asnumpy(prob) if isinstance(prob,cp.ndarray) else np.asarray(prob)
            del gpu,prob
            return ret
        except (RuntimeError,ValueError,TypeError) as exc:
            print('GPU prediction fallback (training stays on GPU):',str(exc)[:250])
    # Warning about mismatched devices is performance-only, not a correctness issue.
    return model.predict_proba(X)[:,1]

print('Checkpoint directory:',PROGRESS_DIR)
print('CuPy:',HAS_CUPY)

In [ ]:
# ===== V4 ORIGINAL DATA, EXACT-ROW AUDIT, SAFE TEACHERS =====
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

# Chỉ tìm CSV dữ liệu gốc; không vô tình lấy train/test cuộc thi.
def find_original_csvs():
    if ORIGINAL_DATASET_DIR:
        roots = [Path(ORIGINAL_DATASET_DIR)]
    else:
        roots = sorted([
            p for p in Path('/kaggle/input').rglob('*') if p.is_dir() and
            ('binary-aviation-satisfaction-129k' in str(p).lower() or
             'airline-passenger-satisfaction' in str(p).lower())
        ], key=lambda p: len(p.parts))
    files = sorted({str(f) for root in roots if root.is_dir() for f in root.rglob('*.csv')})
    if not files:
        raise FileNotFoundError('Add Input original: arseniyshutko/binary-aviation-satisfaction-129k')
    return [Path(p) for p in files]

ORIG_FILES = find_original_csvs()
parts = []
for p in ORIG_FILES:
    d = pd.read_csv(p, low_memory=False)
    d.columns = d.columns.str.strip()
    if ('satisfaction' in d.columns) and all(f in d.columns for f in FEATURES):
        parts.append(d[FEATURES + ['satisfaction']].copy())
if not parts:
    raise RuntimeError('Original CSV không đủ FEATURES và satisfaction!')
original = pd.concat(parts, ignore_index=True)
original = original.drop_duplicates(subset=FEATURES + ['satisfaction']).reset_index(drop=True)
lab = original['satisfaction'].astype(str).str.strip().str.lower().map({
    'satisfied':1, 'neutral or dissatisfied':0, '1':1, '0':0,
    'true':1, 'false':0, 'yes':1, 'no':0, '1.0':1, '0.0':0
})
if lab.isna().any():
    raise ValueError('Original label unknown: ' + str(original.loc[lab.isna(), 'satisfaction'].value_counts().head()))
Y_ORIG = lab.to_numpy(dtype=np.int32)
for c in CATS:
    original[c] = original[c].astype('string').str.strip().fillna('__MISSING__').astype(str)
for c in NUMS:
    original[c] = pd.to_numeric(original[c], errors='coerce')

# DATA-SAFETY: loại original row có 21 features khớp hoàn toàn với train/test cuộc thi.
# Tránh vô tình huấn luyện teacher hay augment trên nhãn của chính sample cần chấm.
def normalized_features(df):
    z = df[FEATURES].copy()
    for c in CATS:
        z[c] = z[c].astype('string').str.strip().fillna('__MISSING__').astype(str)
    for c in NUMS:
        z[c] = pd.to_numeric(z[c], errors='coerce').astype('float64')
    return z

orig_norm = normalized_features(original)
comp_norm = normalized_features(raw)
orig_h = pd.util.hash_pandas_object(orig_norm, index=False).to_numpy(dtype=np.uint64)
comp_h = pd.util.hash_pandas_object(comp_norm, index=False).to_numpy(dtype=np.uint64)
# Hash collision xác suất rất nhỏ. Nếu có rất nhiều matches, kiểm tra thủ công trước khi kết luận.
overlap = np.isin(orig_h, comp_h)
original_overlap_n = int(overlap.sum())
print('AUDIT: original exact feature hash overlap with COMPETITION train+test:', original_overlap_n)
if original_overlap_n:
    original = original.loc[~overlap].reset_index(drop=True)
    Y_ORIG = Y_ORIG[~overlap]
    print('Excluded all overlapping original rows from EVERY v4 teacher and augmentation.')
assert len(original)>50000, ('Unexpected original after overlap removal', len(original))
assert len(original)==len(Y_ORIG)
N_ORIG=len(original)
print('Safe original rows:',N_ORIG,'positive rate:',round(float(Y_ORIG.mean()),5))

# Same 21 feature inputs; include unlabeled competition rows for consistent one-hot schema.
teacher_df = pd.concat([original[FEATURES], raw[FEATURES]],ignore_index=True)
for c in CATS:
    teacher_df[c]=teacher_df[c].astype('string').str.strip().fillna('__MISSING__').astype(str)
for c in NUMS:
    teacher_df[c]=pd.to_numeric(teacher_df[c],errors='coerce')
tmat = pd.get_dummies(teacher_df,columns=CATS,dummy_na=True,dtype=np.float32).to_numpy(dtype=np.float32)
X_ORIG = tmat[:N_ORIG]
X_COMP = tmat[N_ORIG:]
assert X_COMP.shape[0]==len(raw)

# XGB teacher (two independent seeds, trained ONLY on SAFE original labels).
xgb_t_list=[]
xgb_t_info=[]
for seed in TEACHER_SEEDS:
    a,b=train_test_split(np.arange(N_ORIG),test_size=.15,stratify=Y_ORIG,random_state=seed)
    sel=XGBClassifier(n_estimators=2500,max_depth=6,learning_rate=.04,
        min_child_weight=5,subsample=.9,colsample_bytree=.9,reg_lambda=4,
        objective='binary:logistic',eval_metric='auc',tree_method='hist',device='cuda:0',
        n_jobs=4,random_state=seed,early_stopping_rounds=100)
    sel.fit(X_ORIG[a],Y_ORIG[a],eval_set=[(X_ORIG[b],Y_ORIG[b])],verbose=False)
    best=int(sel.best_iteration)+1
    del sel
    full=XGBClassifier(n_estimators=best,max_depth=6,learning_rate=.04,
        min_child_weight=5,subsample=.9,colsample_bytree=.9,reg_lambda=4,
        objective='binary:logistic',eval_metric='auc',tree_method='hist',device='cuda:0',
        n_jobs=4,random_state=seed)
    full.fit(X_ORIG,Y_ORIG,verbose=False)
    pp=np.asarray(predict_proba(full,X_COMP),dtype=np.float32)
    assert np.isfinite(pp).all()
    xgb_t_list.append(pp)
    xgb_t_info.append({'seed':seed,'best_trees':best})
    del full,pp
    gc.collect()
    print('XGB original teacher seed',seed,'trees',best,flush=True)
PX = np.mean(np.stack(xgb_t_list),axis=0).astype(np.float32)

# CatBoost teacher provides truly different original-data representation.
cat_seed=1729
a,b=train_test_split(np.arange(N_ORIG),test_size=.15,stratify=Y_ORIG,random_state=cat_seed)
cat_params = dict(
    iterations=1800, depth=6, learning_rate=.05, l2_leaf_reg=5,
    loss_function='Logloss',eval_metric='AUC',task_type='GPU',devices='0',
    verbose=False,random_seed=cat_seed,allow_writing_files=False,
    gpu_ram_part=0.75,
)
cat_sel=CatBoostClassifier(**cat_params)
cat_sel.fit(X_ORIG[a],Y_ORIG[a],eval_set=(X_ORIG[b],Y_ORIG[b]),early_stopping_rounds=110,verbose=False)
cat_trees=max(1,int(cat_sel.best_iteration_)+1)
del cat_sel; gc.collect()
cat_full=CatBoostClassifier(**{**cat_params,'iterations':cat_trees})
cat_full.fit(X_ORIG,Y_ORIG,verbose=False)
PC=np.asarray(cat_full.predict_proba(X_COMP)[:,1],dtype=np.float32)
del cat_full; gc.collect()

assert PX.shape==PC.shape==(len(raw),)
assert np.isfinite(PX).all() and np.isfinite(PC).all()
def proba_logit(p):
    q=np.clip(p,1e-5,1-1e-5)
    return np.column_stack([p,np.log(q/(1-q))]).astype(np.float32)
SAFE_XGB_TEACHER=proba_logit(PX)
SAFE_DUAL_TEACHER=np.column_stack([proba_logit(PX),proba_logit(PC)]).astype(np.float32)
print('Audited XGB-only teacher competition OOF-like AUC:',float(roc_auc_score(Y,PX[:n_train])))
print('Audited CatBoost teacher competition OOF-like AUC:',float(roc_auc_score(Y,PC[:n_train])))
RUN_META['original_dataset']=[str(p) for p in ORIG_FILES]
RUN_META['original_rows_pre_filter']=int(N_ORIG+original_overlap_n)
RUN_META['original_rows_overlap_excluded']=original_overlap_n
RUN_META['original_rows_safe']=N_ORIG
RUN_META['xgb_original_teachers']=xgb_t_info
RUN_META['cat_original_teacher_trees']=cat_trees
RUN_META['cat_teacher_auc_on_competition']=float(roc_auc_score(Y,PC[:n_train]))
RUN_META['xgb_teacher_auc_on_competition']=float(roc_auc_score(Y,PX[:n_train]))
MANIFEST.write_text(json.dumps(RUN_META,indent=2,ensure_ascii=False),encoding='utf-8')
print('Teachers ready:',SAFE_XGB_TEACHER.shape, SAFE_DUAL_TEACHER.shape)
# Release unnecessary features; original retained for supervised augmentation below.
del orig_norm,comp_norm,orig_h,comp_h,overlap,tmat,teacher_df,X_ORIG,X_COMP,lab,parts,xgb_t_list
import gc;gc.collect()

In [ ]:
# ===== ORIGINAL-ROW AUGMENTATION: LABEL-SAFE FEATURE PREPARATION =====
# Không tự dùng nhãn original để target-encode original rows. Teacher không sử dụng
# khi augment original bởi prediction in-sample có thể khiến model học sai lệch.
# Chỉ dùng nếu RUN_XGB_ORIGAUG True.

def make_context_te(df):
    v=pd.DataFrame(index=range(len(df)))
    for c in FEATURES:
        v[c]=df[c].astype('string').fillna('__MISSING__').astype(str).to_numpy()
    seg=v['Type of Travel']+'|'+v['Customer Type']+'|'+v['Class']
    v['segment_TCC']=seg
    for col in SCORES:
        v['ctx_'+col]=seg+'|'+v[col]
    for col,g in [
        ('Flight Distance','Class'),('Flight Distance','Type of Travel'),
        ('Flight Distance','Customer Type'),('Age','Class'),('Age','Type of Travel')]:
        v['ctx_'+col+'|'+g]=v[col]+'|'+v[g]
    return v

# Keep the original row order unchanged; in this matrix [original rows, competition train, competition test].
if RUN_XGB_ORIGAUG:
    all_raw=pd.concat([original[FEATURES],raw[FEATURES]],ignore_index=True)
    all_base_df=pd.get_dummies(all_raw,columns=CATS,dummy_na=True,dtype=np.float32)
    AUG_BASE=all_base_df.to_numpy(dtype=np.float32)
    AUG_ORIG_BASE=AUG_BASE[:N_ORIG]
    AUG_COMP_BASE=AUG_BASE[N_ORIG:]
    del all_base_df,AUG_BASE

    ORIG_TE_FRAME=make_context_te(original[FEATURES].reset_index(drop=True))
    assert list(ORIG_TE_FRAME.columns)==list(TE_CONTEXT.columns), 'TE columns misaligned!'

    # Unlabeled route stats, computed from all features (original, contest train, contest test).
    # Exclude current row from group aggregates.
    r=all_raw['Flight Distance']
    route_cnt=r.groupby(r,dropna=False).transform('size').to_numpy(dtype=np.float32)
    route_dict={'route_log_count':np.log1p(route_cnt).astype(np.float32)}
    rich_dict={}
    for col in ['Online boarding','Inflight wifi service','Seat comfort',
                'Inflight entertainment','Cleanliness','Ease of Online booking']:
        rate=pd.to_numeric(all_raw[col],errors='coerce').astype(np.float32)
        x=rate.to_numpy(dtype=np.float32)
        sums=rate.groupby(r,dropna=False).transform('sum').to_numpy(dtype=np.float32)
        sq=rate.pow(2).groupby(r,dropna=False).transform('sum').to_numpy(dtype=np.float32)
        den=np.maximum(route_cnt-1,1)
        mean=np.where(route_cnt>1,(sums-x)/den,float(rate.mean())).astype(np.float32)
        route_dict['route_mean_'+col]=mean
        route_dict['route_resid_'+col]=(x-mean).astype(np.float32)
        var=np.where(route_cnt>2,(sq-x*x)/den-mean*mean,0).astype(np.float32)
        rich_dict['route_std_loo_'+col]=np.sqrt(np.maximum(var,0)).astype(np.float32)
        flag=(rate>=4).astype(np.float32)
        high=flag.groupby(r,dropna=False).transform('sum').to_numpy(dtype=np.float32)
        rich_dict['route_high_share_loo_'+col]=np.where(route_cnt>1,(high-flag.to_numpy())/den,float(flag.mean())).astype(np.float32)
    for col in CATS:
        catcounts=all_raw.groupby(['Flight Distance',col],dropna=False)[col].transform('size').to_numpy(dtype=np.float32)
        rich_dict['route_samecat_share_loo_'+col]=np.where(route_cnt>1,(catcounts-1)/np.maximum(route_cnt-1,1),0).astype(np.float32)
    AUG_ROUTE=np.column_stack([pd.DataFrame(route_dict).to_numpy(dtype=np.float32),
                               pd.DataFrame(rich_dict).to_numpy(dtype=np.float32)])
    assert AUG_ROUTE.shape==(N_ORIG+len(raw),29),AUG_ROUTE.shape
    assert np.isfinite(AUG_ROUTE).all()
    AUG_ORIG_ROUTE=AUG_ROUTE[:N_ORIG]
    AUG_COMP_ROUTE=AUG_ROUTE[N_ORIG:]
    del all_raw,route_dict,rich_dict,AUG_ROUTE,r,route_cnt
    gc.collect()
    print('Original augmentation features:',AUG_ORIG_BASE.shape,AUG_ORIG_ROUTE.shape)

In [ ]:
# ===== 4 CLEAN MEMBERS: XGB SAFE CONTROL / XGB DUAL / CAT DUAL / XGB ORIGINAL AUG =====

def cat_student(n_estimators,seed,early_stop=False):
    return CatBoostClassifier(
        iterations=int(n_estimators),depth=7,learning_rate=0.045,
        l2_leaf_reg=6,random_strength=.5,
        loss_function='Logloss',eval_metric='AUC',
        task_type='GPU',devices='0',gpu_ram_part=.75,
        verbose=False,allow_writing_files=False,
        random_seed=int(seed),
    )

CONFIGS = [
    dict(name='quan_xgb_safe_teacher_v4', algo='xgb', aug=False,
         extra=np.column_stack([ROUTE_ARRAY,RICH_EXTRA,SAFE_XGB_TEACHER]),
         desc='v3-context richroute, but original exact-collision removal; XGB teacher'),
    dict(name='quan_xgb_dual_teacher_v4', algo='xgb', aug=False,
         extra=np.column_stack([ROUTE_ARRAY,RICH_EXTRA,SAFE_DUAL_TEACHER]),
         desc='v3-context richroute, audited XGB+Cat original teachers'),
    dict(name='quan_cat_dual_teacher_v4', algo='cat', aug=False,
         extra=np.column_stack([ROUTE_ARRAY,RICH_EXTRA,SAFE_DUAL_TEACHER]),
         desc='CatBoost GPU student, independently train with XGB+Cat original teachers'),
]
if RUN_XGB_ORIGAUG:
    CONFIGS.append(dict(name='quan_xgb_original_aug_v4',algo='xgb',aug=True,
        desc='XGB context richroute; original rows at weight 0.35, no in-sample teacher'))

enabled={
    'quan_xgb_safe_teacher_v4':RUN_SAFE_BASE,
    'quan_xgb_dual_teacher_v4':RUN_XGB_DUAL,
    'quan_cat_dual_teacher_v4':RUN_CAT_DUAL,
    'quan_xgb_original_aug_v4':RUN_XGB_ORIGAUG,
}

def fit_one(name,model_cfg):
    tic=time.perf_counter()
    algo=model_cfg['algo']
    aug=bool(model_cfg['aug'])
    # Signature locks resuming to these exact features/config version, not just filename.
    cfgsig = 'v4-20261017-samefold-inner5TE-internalES-origmask'+str(original_overlap_n)+('-origW0.35' if aug else '-noaug')+'-'+name
    oof,tsum,done,details=open_progress(name,cfgsig)
    print('\nSTART',name,'completed folds',done,flush=True)
    for k in range(5):
        if k in done:continue
        begin=time.perf_counter()
        tr=np.flatnonzero(FOLDS!=k)
        va=np.flatnonzero(FOLDS==k)
        fit,es=train_test_split(tr,test_size=.10,stratify=Y[tr],random_state=42+k)
        # Predefined features; TE learns labels only from competition fit subset.
        enc=encoder(42+k)
        te_fit=enc.fit_transform(TE_CONTEXT.iloc[fit],Y[fit]).astype(np.float32)
        te_es=enc.transform(TE_CONTEXT.iloc[es]).astype(np.float32)
        if aug:
            t_orig=enc.transform(ORIG_TE_FRAME).astype(np.float32)
            X_orig_inner=np.column_stack([AUG_ORIG_BASE,t_orig,AUG_ORIG_ROUTE]).astype(np.float32)
            X_fit=np.column_stack([AUG_COMP_BASE[fit],te_fit,AUG_COMP_ROUTE[fit]]).astype(np.float32)
            X_es=np.column_stack([AUG_COMP_BASE[es],te_es,AUG_COMP_ROUTE[es]]).astype(np.float32)
            wfit=np.concatenate([np.ones(len(fit),dtype=np.float32),
                                 np.full(N_ORIG,ORIG_WEIGHT,dtype=np.float32)])
            X_fit=np.vstack([X_fit,X_orig_inner])
            yfit=np.concatenate([Y[fit],Y_ORIG])
        else:
            extra=model_cfg['extra']
            X_fit=np.column_stack([BASE_ARRAY[fit],te_fit,extra[fit]]).astype(np.float32)
            X_es=np.column_stack([BASE_ARRAY[es],te_es,extra[es]]).astype(np.float32)
            yfit=Y[fit]
            wfit=None
        del enc,te_fit,te_es
        if algo=='xgb':
            selector=learner(MAX_TREES,42+k,True)
            selector.fit(X_fit,yfit,sample_weight=wfit,eval_set=[(X_es,Y[es])],verbose=False)
            best=int(selector.best_iteration)+1
        elif algo=='cat':
            selector=cat_student(2400,42+k,True)
            selector.fit(X_fit,yfit,eval_set=(X_es,Y[es]),early_stopping_rounds=120,verbose=False)
            best=int(selector.best_iteration_)+1
        else:raise ValueError(algo)
        del selector,X_fit,X_es,yfit,wfit
        gc.collect()
        print('fold',k,'selected',best,'trees',flush=True)

        # Refit: outer training folds only; outer validation labels never enter encoder/model.
        enc=encoder(42+k)
        te_tr=enc.fit_transform(TE_CONTEXT.iloc[tr],Y[tr]).astype(np.float32)
        te_va=enc.transform(TE_CONTEXT.iloc[va]).astype(np.float32)
        te_test=enc.transform(TE_CONTEXT.iloc[n_train:]).astype(np.float32)
        if aug:
            te_orig=enc.transform(ORIG_TE_FRAME).astype(np.float32)
            Xtr=np.column_stack([AUG_COMP_BASE[tr],te_tr,AUG_COMP_ROUTE[tr]]).astype(np.float32)
            Xva=np.column_stack([AUG_COMP_BASE[va],te_va,AUG_COMP_ROUTE[va]]).astype(np.float32)
            Xte=np.column_stack([AUG_COMP_BASE[n_train:],te_test,AUG_COMP_ROUTE[n_train:]]).astype(np.float32)
            Xo=np.column_stack([AUG_ORIG_BASE,te_orig,AUG_ORIG_ROUTE]).astype(np.float32)
            Xtr=np.vstack([Xtr,Xo])
            ytr=np.concatenate([Y[tr],Y_ORIG])
            weights=np.concatenate([np.ones(len(tr),dtype=np.float32),
                                    np.full(N_ORIG,ORIG_WEIGHT,dtype=np.float32)])
            del Xo,te_orig
        else:
            extra=model_cfg['extra']
            Xtr=np.column_stack([BASE_ARRAY[tr],te_tr,extra[tr]]).astype(np.float32)
            Xva=np.column_stack([BASE_ARRAY[va],te_va,extra[va]]).astype(np.float32)
            Xte=np.column_stack([BASE_ARRAY[n_train:],te_test,extra[n_train:]]).astype(np.float32)
            ytr=Y[tr]
            weights=None
        del enc,te_tr,te_va,te_test
        gc.collect()

        if algo=='xgb':
            model=learner(best,42+k,False)
            model.fit(Xtr,ytr,sample_weight=weights,verbose=False)
            pva=np.asarray(predict_proba(model,Xva),dtype=np.float64)
            pte=np.asarray(predict_proba(model,Xte),dtype=np.float64)
        else:
            model=cat_student(best,42+k)
            model.fit(Xtr,ytr,verbose=False)
            pva=np.asarray(model.predict_proba(Xva)[:,1],dtype=np.float64)
            pte=np.asarray(model.predict_proba(Xte)[:,1],dtype=np.float64)
        assert pva.shape==(len(va),) and pte.shape==(len(test),)
        assert np.isfinite(pva).all() and np.isfinite(pte).all()
        assert np.all((pva>=0)&(pva<=1)) and np.all((pte>=0)&(pte<=1))
        oof[va]=pva
        tsum+=pte
        score=float(roc_auc_score(Y[va],pva))
        details.append(dict(fold=k,auc=score,best_trees=best,
                            minutes=round((time.perf_counter()-begin)/60,2)))
        done.append(k)
        save_progress(name,cfgsig,oof,tsum,done,details)
        print('fold',k,'AUC',round(score,9),'checkpoint saved',flush=True)
        del model,Xtr,Xva,Xte,pva,pte,ytr,weights
        gc.collect()

    assert sorted(done)==list(range(5))
    probs=tsum/5
    official_save(name,oof,probs)
    auc=float(roc_auc_score(Y,oof))
    RUN_META['models'][name]={
       'complete':True,'overall_oof_auc':auc,
       'delta_vs_v3':auc-REFERENCE_V3,
       'fold_aucs':fold_summary(oof),'fold_details':details,
       'minutes':round((time.perf_counter()-tic)/60,2),
       'experiment':model_cfg['desc']}
    experiment_row(name,auc,model_cfg['desc'])
    package_results()
    print('DONE',name,'OOF AUC',round(auc,10),'delta vs v3',round(auc-REFERENCE_V3,10),flush=True)

for cfg in CONFIGS:
    if enabled[cfg['name']]:
        fit_one(cfg['name'],cfg)

package_results()
with zipfile.ZipFile(OUT_ZIP) as z:
    for c in CONFIGS:
        if not enabled[c['name']]:continue
        assert 'oof_'+c['name']+'.npy' in z.namelist()
        assert 'test_'+c['name']+'.npy' in z.namelist()
print('\nRESULT SUMMARY (best v3:',REFERENCE_V3,')')
for name,d in RUN_META['models'].items():
    if d.get('complete'):
        print(name,'OOF',format(d['overall_oof_auc'],'.10f'),
              'delta_v3',format(d['overall_oof_auc']-REFERENCE_V3,'+.10f'))
print('MANIFEST',MANIFEST)
print('ZIP',OUT_ZIP)

## Sau khi hoàn tất

Mở `S6E10_Power_v4_manifest.json` để xem `overall_oof_auc`, `fold_aucs`, `original_rows_overlap_excluded` và `complete`. ZIP `S6E10_Power_v4_members.zip` chứa `oof_<member>.npy` và `test_<member>.npy` đúng thứ tự competition.

Để xác nhận mô hình mới có ích, đề nghị **người phụ trách stacking** so sánh đóng góp vào stack CV với các model v3; không nộp/blend riêng theo vai trò member. Nếu thời gian GPU ít, có thể đặt `RUN_CAT_DUAL=False` hoặc `RUN_XGB_ORIGAUG=False` ở cell đầu.